In [1]:

import csv
from typing import Tuple

import numpy as np
from quri_parts.algo.optimizer import Adam
from quri_parts.core.estimator.gradient import (
    create_numerical_gradient_estimator,
)
from quri_parts.qulacs.estimator import (
    create_qulacs_vector_concurrent_estimator,
    create_qulacs_vector_concurrent_parametric_estimator,
)
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from scikit_quri.circuit import LearningCircuit
from scikit_quri.circuit.pre_defined import create_dqn_cl, create_dqn_cl_no_cz
from scikit_quri.qnn.classifier import QNNClassifier
# from scikit_quri.qnn.solver import Adam

In [2]:
# Use wine dataset retrieved from: https://archive-beta.ics.uci.edu/ml/datasets/wine
def load_dataset(
    file_path: str, ignore_kind: int, test_ratio: float,
) -> Tuple[np.array, np.array, np.array, np.array]:
    """Load dataset from specified path.

    Args:
        file_path: File path from which data is loaded.
        ignore_kind: The dataset expected to have 3 classes and we need 2 classes to test. So specify here which class to ignore in loading.

    """
    x = []
    y = []
    with open(file_path) as f:
        reader = csv.reader(f)
        for row in reader:
            kind = int(row[0])
            if kind == ignore_kind:
                continue
            y.append(kind)
            x.append([float(feature) for feature in row[1:]])

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=test_ratio, shuffle=True)

    return x_train, x_test, y_train, y_test

In [3]:
from scikit_quri.backend import SimEstimator
def create_classifier(n_features: int, circuit: LearningCircuit, locality: int):
    adam = Adam()
    estimator = SimEstimator()
    gradient_estimator = create_numerical_gradient_estimator(
        create_qulacs_vector_concurrent_parametric_estimator(), delta=1e-3,
    )
    classifier = QNNClassifier(circuit, 2, estimator, gradient_estimator, optimizer=adam)
    return classifier

In [4]:
import cProfile
x_train, x_test, y_train, y_test = load_dataset("../datasets/wine.data", 3, 0.5)
for i in range(len(y_train)):
    y_train[i] -= 1
for i in range(len(y_test)):
    y_test[i] -= 1

n_features = 13
locality = 2
maxiter = 5

n_train = 10
n_test = 10
x_train = x_train[:n_train]
y_train = y_train[:n_train]
x_test = x_test[:n_test]
y_test = y_test[:n_test]

profiler = cProfile.Profile()
print("CL")
circuit = create_dqn_cl(n_features, 5, locality)
classifier = create_classifier(n_features, circuit, locality)
result_cl = []
loop_size = 80
for i in range(loop_size):
    classifier.fit(np.array(x_train), np.array(y_train), maxiter)
    y_pred = classifier.predict(np.array(x_test)).argmax(axis=1)
    score = f1_score(y_test, y_pred, average="weighted")
    result_cl.append(score)
    print(f"iteration:{(i + 1) * maxiter} score:{score}")

print("no CL")
circuit = create_dqn_cl_no_cz(n_features, 5)
classifier = create_classifier(n_features, circuit, locality)
result_no_cl = []
for i in range(loop_size):
    classifier.fit(np.array(x_train), np.array(y_train), maxiter)
    y_pred = classifier.predict(np.array(x_test)).argmax(axis=1)
    score = f1_score(y_test, y_pred, average="weighted")
    result_no_cl.append(score)
    print(f"iteration:{(i + 1) * maxiter} score:{score}")

CL
 iter:4/5 cost:optimizer_state.cost=0.5396674390324828
iteration:5 score:0.6703296703296703
 iter:4/5 cost:optimizer_state.cost=0.41041651230168175
iteration:10 score:0.898989898989899
 iter:4/5 cost:optimizer_state.cost=0.31131032829834865
iteration:15 score:1.0
 iter:4/5 cost:optimizer_state.cost=0.24645272636886673
iteration:20 score:1.0
 iter:4/5 cost:optimizer_state.cost=0.21163598246409066
iteration:25 score:1.0
 iter:4/5 cost:optimizer_state.cost=0.19697003616157172
iteration:30 score:1.0
 iter:4/5 cost:optimizer_state.cost=0.18871907705703764
iteration:35 score:1.0
 iter:4/5 cost:optimizer_state.cost=0.18073637453918015
iteration:40 score:1.0
 iter:4/5 cost:optimizer_state.cost=0.17308058431810208
iteration:45 score:1.0
 iter:4/5 cost:optimizer_state.cost=0.16868505252229063
iteration:50 score:1.0
 iter:4/5 cost:optimizer_state.cost=0.16552529732889853
iteration:55 score:1.0
 iter:4/5 cost:optimizer_state.cost=0.16199204935107332
iteration:60 score:1.0
 iter:4/5 cost:optimiz

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

plt.plot(np.arange(0, loop_size * maxiter, step=maxiter), np.array(result_cl).flatten(), label="CL")
plt.plot(
    np.arange(0, loop_size * maxiter, step=maxiter), np.array(result_no_cl).flatten(), label="no CL",
)
plt.xticks(np.arange(0, loop_size * maxiter, step=25))
plt.xlabel("iteration")
plt.ylabel("score")
plt.legend()
plt.show()